In [66]:
import pandas as pd
import os
from datetime import datetime
from meteostat import Point, Monthly

# ---------------------------------------------------------
# 1. AYARLAR
# ---------------------------------------------------------

file_path = 'worldcities.csv'
log_file_path = 'hatali_sehirler_log.txt'

# Avrupa Ülkeleri Listesi (ISO2 Kodları - worldcities.csv 'iso2' kolonu ile eşleşir)
europe_codes = [
    'TR', 'RU', 'DE', 'FR', 'GB', 'IT', 'ES', 'UA', 'PL', 'RO', 
    'NL', 'BE', 'CZ', 'GR', 'PT', 'SE', 'HU', 'AT', 'BY', 'CH', 
    'BG', 'RS', 'DK', 'FI', 'SK', 'NO', 'IE', 'HR', 'MD', 'BA', 
    'AL', 'LT', 'MK', 'SI', 'LV', 'EE', 'ME', 'LU', 'MT', 'IS', 
    'AD', 'MC', 'LI', 'SM', 'VA', 'XK'
]

# Tarih Aralığı
start = datetime(2012, 1, 1)
end = datetime.now()

# ---------------------------------------------------------
# 2. LOGLAMA FONKSİYONU
# ---------------------------------------------------------
def log_error(city_row, error_message):
    timestamp = datetime.now().strftime("%Y-%m-%d %H:%M:%S")
    # CSV'deki kolon isimlerine göre loglama
    log_line = f"{timestamp} | ID: {city_row['id']} | City: {city_row['city']} | Country: {city_row['iso2']} | Error: {error_message}\n"
    
    try:
        with open(log_file_path, "a", encoding="utf-8") as f:
            f.write(log_line)
    except Exception as e:
        print(f"Log dosyasına yazılamadı: {e}")

# ---------------------------------------------------------
# 3. VERİYİ OKUMA VE İŞLEME
# ---------------------------------------------------------

# Log dosyasını sıfırla
with open(log_file_path, "w", encoding="utf-8") as f:
    f.write(f"--- İŞLEM BAŞLANGICI: {datetime.now()} ---\n")
    f.write("Log Formatı: ZAMAN | ID | ŞEHİR | ÜLKE KODU | HATA\n")
    f.write("-" * 80 + "\n")

all_weather_data = []

if not os.path.exists(file_path):
    print(f"HATA: Dosya bulunamadı: {file_path}")
else:
    print(f"CSV okunuyor: {file_path}...")
    
    # CSV dosyasını Pandas ile oku
    df_cities = pd.read_csv(file_path)
    
    # Sadece Avrupa ülkelerini (europe_codes içindekileri) filtrele
    # iso2 kolonu ülke kodlarını tutuyor
    europe_cities = df_cities[df_cities['iso2'].isin(europe_codes)].copy()
    
    total_cities = len(europe_cities)
    print(f"Toplam Şehir Sayısı (Dünya): {len(df_cities)}")
    print(f"Filtrelenen Avrupa Şehir Sayısı: {total_cities}")
    print("Veri çekme işlemi başlıyor...\n")

    count = 0
    success_count = 0
    
    # Filtrelenmiş DataFrame üzerinde döngü
    for index, row in europe_cities.iterrows():
        try:
            # CSV kolon isimleri: city, lat, lng, iso2, id
            lat = row['lat']
            lng = row['lng'] # Görselde 'lng' olarak görünüyor
            city_name = row['city']
            country_iso = row['iso2']
            city_id = row['id']

            # Meteostat Point oluştur
            location = Point(lat, lng)
            
            # Veriyi çek
            data = Monthly(location, start, end)
            data = data.fetch()

            if not data.empty:
                # Meta verileri ekle
                data['city_id'] = city_id
                data['city_name'] = city_name
                data['country'] = country_iso
                data['lat'] = lat
                data['lng'] = lng
                
                # İndeksi sütuna çevir (time sütunu oluşur)
                data.reset_index(inplace=True)
                
                # İstenen sütunları seç
                selected_cols = data[['time', 'city_id', 'city_name', 'country', 'tavg', 'tmin', 'tmax', 'lat', 'lng']]
                all_weather_data.append(selected_cols)
                
                success_count += 1
            else:
                # Meteostat istasyon bulamadı
                log_error(row, "Meteostat verisi boş (Empty Data)")

        except Exception as e:
            # Beklenmeyen bir hata
            log_error(row, f"Hata: {str(e)}")
        
        count += 1
        # İlerleme durumunu göster (Her 50 şehirde bir)
        if count % 50 == 0:
            print(f"İlerleme: {count}/{total_cities} şehir tarandı. (Başarılı: {success_count})")

    # ---------------------------------------------------------
    # 4. SONUCU GÖSTERME
    # ---------------------------------------------------------

    print("\n" + "="*50)
    print("İŞLEM TAMAMLANDI")
    print(f"Toplam Taranan: {count}")
    print(f"Başarılı Veri: {success_count}")
    print(f"Hatalar '{log_file_path}' dosyasına kaydedildi.")
    print("="*50)

    if all_weather_data:
        # Tüm verileri birleştir
        final_df = pd.concat(all_weather_data, ignore_index=True)
        
        print("\nSONUÇ DATAFRAME (İLK 20 SATIR):")
        print(final_df.head(20))
        
        print("\nDATAFRAME BİLGİSİ:")
        print(final_df.info())
    else:
        print("Hiçbir veri çekilemedi.")

CSV okunuyor: worldcities.csv...
Toplam Şehir Sayısı (Dünya): 48059
Filtrelenen Avrupa Şehir Sayısı: 13044
Veri çekme işlemi başlıyor...



İlerleme: 50/13044 şehir tarandı. (Başarılı: 50)


İlerleme: 100/13044 şehir tarandı. (Başarılı: 99)


İlerleme: 150/13044 şehir tarandı. (Başarılı: 147)


İlerleme: 200/13044 şehir tarandı. (Başarılı: 196)


...

İlerleme: 9950/13044 şehir tarandı. (Başarılı: 8550)
İlerleme: 10000/13044 şehir tarandı. (Başarılı: 8587)


İlerleme: 10050/13044 şehir tarandı. (Başarılı: 8631)
İlerleme: 10100/13044 şehir tarandı. (Başarılı: 8677)


İlerleme: 10150/13044 şehir tarandı. (Başarılı: 8719)
İlerleme: 10200/13044 şehir tarandı. (Başarılı: 8762)
İlerleme: 10250/13044 şehir tarandı. (Başarılı: 8805)
İlerleme: 10300/13044 şehir tarandı. (Başarılı: 8840)


İlerleme: 10350/13044 şehir tarandı. (Başarılı: 8883)
İlerleme: 10400/13044 şehir tarandı. (Başarılı: 8929)
İlerleme: 10450/13044 şehir tarandı. (Başarılı: 8973)
İlerleme: 10500/13044 şehir tarandı. (Başarılı: 9012)
İlerleme: 10550/13044 şehir tarandı. (Başarılı: 9056)
İlerleme: 10600/13044 şehir tarandı. (Başarılı: 9099)
İlerleme: 10650/13044 şehir tarandı. (Başarılı: 9144)
İlerleme: 10700/13044 şehir tarandı. (Başarılı: 9182)
İlerleme: 10750/13044 şehir tarandı. (Başarılı: 9227)
İlerleme: 10800/13044 şehir tarandı. (Başarılı: 9265)
İlerleme: 10850/13044 şehir tarandı. (Başarılı: 9307)
İlerleme: 10900/13044 şehir tarandı. (Başarılı: 9347)
İlerleme: 10950/13044 şehir tarandı. (Başarılı: 9389)
İlerleme: 11000/13044 şehir tarandı. (Başarılı: 9430)
İlerleme: 11050/13044 şehir tarandı. (Başarılı: 9471)
İlerleme: 11100/13044 şehir tarandı. (Başarılı: 9514)
İlerleme: 11150/13044 şehir tarandı. (Başarılı: 9556)
İlerleme: 11200/13044 şehir tarandı. (Başarılı: 9599)
İlerleme: 11250/13044 şehir 

İlerleme: 12400/13044 şehir tarandı. (Başarılı: 10634)
İlerleme: 12450/13044 şehir tarandı. (Başarılı: 10672)
İlerleme: 12500/13044 şehir tarandı. (Başarılı: 10713)
İlerleme: 12550/13044 şehir tarandı. (Başarılı: 10754)
İlerleme: 12600/13044 şehir tarandı. (Başarılı: 10799)
İlerleme: 12650/13044 şehir tarandı. (Başarılı: 10839)
İlerleme: 12700/13044 şehir tarandı. (Başarılı: 10887)
İlerleme: 12750/13044 şehir tarandı. (Başarılı: 10930)
İlerleme: 12800/13044 şehir tarandı. (Başarılı: 10974)
İlerleme: 12850/13044 şehir tarandı. (Başarılı: 11017)
İlerleme: 12900/13044 şehir tarandı. (Başarılı: 11063)
İlerleme: 12950/13044 şehir tarandı. (Başarılı: 11109)
İlerleme: 13000/13044 şehir tarandı. (Başarılı: 11136)

İŞLEM TAMAMLANDI
Toplam Taranan: 13044
Başarılı Veri: 11165
Hatalar 'hatali_sehirler_log.txt' dosyasına kaydedildi.

SONUÇ DATAFRAME (İLK 20 SATIR):
         time     city_id city_name country  tavg  tmin  tmax      lat  \
0  2012-01-01  1643318494    Moscow      RU  -6.8  -8.9  -4.5

In [68]:
final_df.to_csv("final.csv", index=False)

In [69]:
final_df

,time,city_id,city_name,country,tavg,tmin,tmax,lat,lng
0,2012-01-01,1643318494,Moscow,RU,-6.8,-8.9,-4.5,55.7506,37.6175
1,2012-02-01,1643318494,Moscow,RU,-11.7,-14.8,-8.6,55.7506,37.6175
2,2012-03-01,1643318494,Moscow,RU,-3.1,-6.4,0.0,55.7506,37.6175
3,2012-04-01,1643318494,Moscow,RU,8.2,3.7,13.0,55.7506,37.6175
4,2012-05-01,1643318494,Moscow,RU,15.1,9.4,20.9,55.7506,37.6175
...,...,...,...,...,...,...,...,...,...
1700018,2025-05-01,1705477555,Zavrč,SI,14.7,8.9,20.1,46.3917,16.0497
1700019,2025-06-01,1705477555,Zavrč,SI,22.8,15.1,29.0,46.3917,16.0497
1700020,2025-07-01,1705477555,Zavrč,SI,22.3,16.0,28.2,46.3917,16.0497
1700021,2025-08-01,1705477555,Zavrč,SI,21.5,14.8,27.9,46.3917,16.0497


In [70]:
import pandas as pd
import os
from datetime import datetime
from meteostat import Point, Monthly

# ---------------------------------------------------------
# 1. AYARLAR
# ---------------------------------------------------------

file_path = 'worldcities.csv'
log_file_path = 'hatali_sehirler_log.txt'
output_csv_path = 'tum_dunya_hava_verileri.csv' # Sonuçların kaydedileceği dosya

# Tarih Aralığı
start = datetime(2012, 1, 1)
end = datetime.now()

# ---------------------------------------------------------
# 2. LOGLAMA FONKSİYONU
# ---------------------------------------------------------
def log_error(city_row, error_message):
    timestamp = datetime.now().strftime("%Y-%m-%d %H:%M:%S")
    # CSV'deki kolon isimlerine göre loglama
    log_line = f"{timestamp} | ID: {city_row['id']} | City: {city_row['city']} | Country: {city_row['iso2']} | Error: {error_message}\n"
    
    try:
        with open(log_file_path, "a", encoding="utf-8") as f:
            f.write(log_line)
    except Exception as e:
        print(f"Log dosyasına yazılamadı: {e}")

# ---------------------------------------------------------
# 3. VERİYİ OKUMA VE İŞLEME
# ---------------------------------------------------------

# Log dosyasını sıfırla
with open(log_file_path, "w", encoding="utf-8") as f:
    f.write(f"--- İŞLEM BAŞLANGICI: {datetime.now()} ---\n")
    f.write("Log Formatı: ZAMAN | ID | ŞEHİR | ÜLKE KODU | HATA\n")
    f.write("-" * 80 + "\n")

all_weather_data = []

if not os.path.exists(file_path):
    print(f"HATA: Dosya bulunamadı: {file_path}")
else:
    print(f"CSV okunuyor: {file_path}...")
    
    # CSV dosyasını Pandas ile oku
    df_cities = pd.read_csv(file_path)
    
    # FİLTRELEME İŞLEMİ KALDIRILDI
    # Doğrudan tüm veri seti üzerinde çalışacağız
    
    total_cities = len(df_cities)
    print(f"Toplam Şehir Sayısı (Tüm Dünya): {total_cities}")
    print("Veri çekme işlemi başlıyor... (Bu işlem uzun sürebilir)\n")

    count = 0
    success_count = 0
    
    # Tüm DataFrame üzerinde döngü
    for index, row in df_cities.iterrows():
        try:
            # CSV kolon isimleri: city, lat, lng, iso2, id
            lat = row['lat']
            lng = row['lng'] 
            city_name = row['city']
            country_iso = row['iso2']
            city_id = row['id']

            # Meteostat Point oluştur
            location = Point(lat, lng)
            
            # Veriyi çek
            data = Monthly(location, start, end)
            data = data.fetch()

            if not data.empty:
                # Meta verileri ekle
                data['city_id'] = city_id
                data['city_name'] = city_name
                data['country'] = country_iso
                data['lat'] = lat
                data['lng'] = lng
                
                # İndeksi sütuna çevir (time sütunu oluşur)
                data.reset_index(inplace=True)
                
                # İstenen sütunları seç
                selected_cols = data[['time', 'city_id', 'city_name', 'country', 'tavg', 'tmin', 'tmax', 'lat', 'lng']]
                all_weather_data.append(selected_cols)
                
                success_count += 1
            else:
                # Meteostat istasyon bulamadı
                log_error(row, "Meteostat verisi boş (Empty Data)")

        except Exception as e:
            # Beklenmeyen bir hata
            log_error(row, f"Hata: {str(e)}")
        
        count += 1
        
        # İlerleme durumunu göster (Her 100 şehirde bir yazdıralım, liste uzun olduğu için)
        if count % 100 == 0:
            print(f"İlerleme: {count}/{total_cities} şehir tarandı. (Başarılı: {success_count})")

    # ---------------------------------------------------------
    # 4. SONUCU GÖSTERME VE KAYDETME
    # ---------------------------------------------------------

    print("\n" + "="*50)
    print("İŞLEM TAMAMLANDI")
    print(f"Toplam Taranan: {count}")
    print(f"Başarılı Veri (Şehir Bazlı): {success_count}")
    print(f"Hatalar '{log_file_path}' dosyasına kaydedildi.")
    print("="*50)

    if all_weather_data:
        # Tüm verileri birleştir
        final_df = pd.concat(all_weather_data, ignore_index=True)
        
        print("\nSONUÇ DATAFRAME (İLK 20 SATIR):")
        print(final_df.head(20))
        
        print("\nDATAFRAME BİLGİSİ:")
        print(final_df.info())

        # Veriyi CSV olarak kaydet (Çok önemli, çünkü veri çok büyük olacak)
        print(f"\nVeriler '{output_csv_path}' dosyasına kaydediliyor...")
        final_df.to_csv(output_csv_path, index=False)
        print("Kayıt işlemi başarılı.")
        
    else:
        print("Hiçbir veri çekilemedi.")

CSV okunuyor: worldcities.csv...
Toplam Şehir Sayısı (Tüm Dünya): 48059
Veri çekme işlemi başlıyor... (Bu işlem uzun sürebilir)



İlerleme: 100/48059 şehir tarandı. (Başarılı: 82)


İlerleme: 200/48059 şehir tarandı. (Başarılı: 144)


İlerleme: 300/48059 şehir tarandı. (Başarılı: 217)
İlerleme: 400/48059 şehir tarandı. (Başarılı: 285)


İlerleme: 500/48059 şehir tarandı. (Başarılı: 363)


...

İlerleme: 47400/48059 şehir tarandı. (Başarılı: 31803)


İlerleme: 47500/48059 şehir tarandı. (Başarılı: 31874)


İlerleme: 47600/48059 şehir tarandı. (Başarılı: 31944)
İlerleme: 47700/48059 şehir tarandı. (Başarılı: 32022)


İlerleme: 47800/48059 şehir tarandı. (Başarılı: 32067)
İlerleme: 47900/48059 şehir tarandı. (Başarılı: 32121)


İlerleme: 48000/48059 şehir tarandı. (Başarılı: 32179)

İŞLEM TAMAMLANDI
Toplam Taranan: 48059
Başarılı Veri (Şehir Bazlı): 32207
Hatalar 'hatali_sehirler_log.txt' dosyasına kaydedildi.

SONUÇ DATAFRAME (İLK 20 SATIR):
         time     city_id city_name country  tavg  tmin  tmax     lat  \
0  2012-01-01  1392685764     Tokyo      JP   4.8   1.8   8.3  35.687   
1  2012-02-01  1392685764     Tokyo      JP   5.4   2.2   9.1  35.687   
2  2012-03-01  1392685764     Tokyo      JP   8.8   5.3  12.5  35.687   
3  2012-04-01  1392685764     Tokyo      JP  14.5  11.0  18.5  35.687   
4  2012-05-01  1392685764     Tokyo      JP  19.6  16.1  23.6  35.687   
5  2012-06-01  1392685764     Tokyo      JP  21.4  18.6  24.8  35.687   
6  2012-07-01  1392685764     Tokyo      JP  26.4  23.5  30.1  35.687   
7  2012-08-01  1392685764     Tokyo      JP  29.1  26.3  33.1  35.687   
8  2012-09-01  1392685764     Tokyo      JP  26.2  23.3  29.8  35.687   
9  2012-10-01  1392685764     Tokyo      JP  19.4  

In [22]:
import pandas as pd
import os
from datetime import datetime
from meteostat import Point, Monthly
from meteostat import Stations

polatli = Point(39.5842, 32.1472)

In [23]:
stations = Stations()
stations = stations.nearby(39.5842, 32.1472)
station = stations.fetch(1)

print(station)

                     name country region    wmo  icao  latitude  longitude  \
id                                                                           
17129  Ankara / Etimesgut      TR    ANK  17129  LTAD     39.95    32.6833   

       elevation         timezone hourly_start hourly_end daily_start  \
id                                                                      
17129      799.0  Europe/Istanbul   1945-01-31 2025-11-22  1945-02-08   

       daily_end monthly_start monthly_end      distance  
id                                                        
17129 2022-04-24    1945-01-01  2021-01-01  61269.530269  


In [16]:
polatli

In [24]:
start = datetime(2000, 1, 1)
end = datetime(2025, 12, 31)

# Get Monthly data
data = Monthly(station, start, end)

In [25]:
data = data.fetch()
data

,tavg,tmin,tmax,prcp,wspd,pres,tsun
time,,,,,,,
2000-06-01,19.2,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>
2000-07-01,25.7,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>
2000-08-01,22.8,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>
2000-09-01,18.2,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>
2000-12-01,1.1,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>
...,...,...,...,...,...,...,...
2025-05-01,16.7,9.3,23.6,70.2,<NA>,1013.3,<NA>
2025-06-01,22.1,13.6,29.2,14.4,<NA>,1015.3,<NA>
2025-07-01,27.1,18.6,34.6,0.9,<NA>,1010.4,<NA>


In [26]:
data

,tavg,tmin,tmax,prcp,wspd,pres,tsun
time,,,,,,,
2000-06-01,19.2,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>
2000-07-01,25.7,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>
2000-08-01,22.8,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>
2000-09-01,18.2,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>
2000-12-01,1.1,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>
...,...,...,...,...,...,...,...
2025-05-01,16.7,9.3,23.6,70.2,<NA>,1013.3,<NA>
2025-06-01,22.1,13.6,29.2,14.4,<NA>,1015.3,<NA>
2025-07-01,27.1,18.6,34.6,0.9,<NA>,1010.4,<NA>


In [27]:
import pandas as pd
import os
import re
from datetime import datetime
from meteostat import Stations, Monthly

# ---------------------------------------------------------
# 1. AYARLAR
# ---------------------------------------------------------

file_path = 'worldcities.csv'
log_file_path = 'hatali_sehirler_log.txt'      # Eski log + yeni hatalar
output_csv_path = 'hatali_sehirler_hava_verileri.csv'

# Tarih Aralığı
start = datetime(2012, 1, 1)
end = datetime.now()

# ---------------------------------------------------------
# 2. ESKİ LOGDAN HATALI ŞEHİR ID'LERİNİ ÇEK
# ---------------------------------------------------------
error_ids = set()

if not os.path.exists(log_file_path):
    print(f"HATA: Hatalı şehirler log dosyası bulunamadı: {log_file_path}")
    raise SystemExit

with open(log_file_path, "r", encoding="utf-8") as f:
    for line in f:
        m = re.search(r"ID:\s*(\d+)", line)
        if m:
            error_ids.add(int(m.group(1)))

if not error_ids:
    print("Log dosyasında hiç ID bulunamadı. Çıkılıyor.")
    raise SystemExit

print(f"Log dosyasından alınan hatalı şehir sayısı: {len(error_ids)}")

# ---------------------------------------------------------
# 3. LOGLAMA FONKSİYONU (YENİ ÇALIŞMA İÇİN)
# ---------------------------------------------------------
def log_error(city_row, error_message):
    timestamp = datetime.now().strftime("%Y-%m-%d %H:%M:%S")
    log_line = (
        f"{timestamp} | ID: {city_row['id']} | City: {city_row['city']} "
        f"| Country: {city_row['iso2']} | Error: {error_message}\n"
    )
    try:
        with open(log_file_path, "a", encoding="utf-8") as f:
            f.write(log_line)
    except Exception as e:
        print(f"Log dosyasına yazılamadı: {e}")

# Eski logun üstüne yazmak istiyorsan burayı kullan;
# sadece eklemek istersen bu kısmı sil.
with open(log_file_path, "w", encoding="utf-8") as f:
    f.write(f"--- YENİ HATALI ŞEHİRLER İŞLEM BAŞLANGICI: {datetime.now()} ---\n")
    f.write("Log Formatı: ZAMAN | ID | ŞEHİR | ÜLKE KODU | HATA\n")
    f.write("-" * 80 + "\n")

# ---------------------------------------------------------
# 4. CSV'Yİ OKU VE SADECE HATALI ŞEHİRLERİ SEÇ
# ---------------------------------------------------------
if not os.path.exists(file_path):
    print(f"HATA: Dosya bulunamadı: {file_path}")
    raise SystemExit

print(f"CSV okunuyor: {file_path}...")
df_cities = pd.read_csv(file_path)

# id türlerini hizala
df_cities['id'] = df_cities['id'].astype(int)

df_error_cities = df_cities[df_cities['id'].isin(error_ids)].copy()

if df_error_cities.empty:
    print("worldcities.csv içinde logdaki ID'lere karşılık gelen şehir bulunamadı.")
    raise SystemExit

total_cities = len(df_error_cities)
print(f"Toplam işlenecek hatalı şehir sayısı: {total_cities}")
print("Veri çekme işlemi başlıyor...\n")

# ---------------------------------------------------------
# 5. METEOSTAT'TAN VERİ ÇEKME (STATION / LOCATION İLE)
# ---------------------------------------------------------
all_weather_data = []
count = 0
success_count = 0

for index, row in df_error_cities.iterrows():
    try:
        lat = row['lat']
        lng = row['lng']
        city_name = row['city']
        country_iso = row['iso2']
        city_id = row['id']

        # En yakın istasyonları bul
        stations = Stations()
        stations = stations.nearby(lat, lng)

        # Tek bir istasyon seç (en yakın)
        station = stations.fetch(1)

        if station.empty:
            log_error(row, "Yakın istasyon bulunamadı")
            continue

        # station DataFrame'ini location olarak kullan
        data = Monthly(station, start, end)
        data = data.fetch()

        if not data.empty:
            # İstasyon ID'si (index) – ilk satır
            station_id = station.index[0]

            data['station_id'] = station_id
            data['city_id'] = city_id
            data['city_name'] = city_name
            data['country'] = country_iso
            data['lat'] = lat
            data['lng'] = lng

            # time index'ini sütuna çevir
            data.reset_index(inplace=True)

            selected_cols = data[
                ['time', 'station_id', 'city_id', 'city_name', 'country',
                 'tavg', 'tmin', 'tmax', 'lat', 'lng']
            ]
            all_weather_data.append(selected_cols)
            success_count += 1
        else:
            log_error(row, "Meteostat verisi boş (Empty Data - station)")

    except Exception as e:
        log_error(row, f"Hata: {str(e)}")

    count += 1

    if count % 20 == 0 or count == total_cities:
        print(f"İlerleme: {count}/{total_cities} şehir tarandı. (Başarılı: {success_count})")

# ---------------------------------------------------------
# 6. SONUCU KAYDET
# ---------------------------------------------------------
print("\n" + "="*50)
print("HATALI ŞEHİRLER İŞLEMİ TAMAMLANDI")
print(f"Toplam Taranan (Hatalı): {count}")
print(f"Başarılı Veri (Şehir Bazlı): {success_count}")
print(f"Hatalar '{log_file_path}' dosyasına kaydedildi.")
print("="*50)

if all_weather_data:
    final_df = pd.concat(all_weather_data, ignore_index=True)
    print("\nİlk 20 satır:")
    print(final_df.head(20))
    print("\nDataFrame bilgisi:")
    print(final_df.info())

    print(f"\nVeriler '{output_csv_path}' dosyasına kaydediliyor...")
    final_df.to_csv(output_csv_path, index=False)
    print("Kayıt işlemi başarılı.")
else:
    print("Hiçbir şehir için veri çekilemedi.")
    

Log dosyasından alınan hatalı şehir sayısı: 15852
CSV okunuyor: worldcities.csv...
Toplam işlenecek hatalı şehir sayısı: 15852
Veri çekme işlemi başlıyor...

İlerleme: 20/15852 şehir tarandı. (Başarılı: 20)
İlerleme: 40/15852 şehir tarandı. (Başarılı: 40)
İlerleme: 60/15852 şehir tarandı. (Başarılı: 60)
İlerleme: 80/15852 şehir tarandı. (Başarılı: 80)
İlerleme: 100/15852 şehir tarandı. (Başarılı: 100)
İlerleme: 120/15852 şehir tarandı. (Başarılı: 120)
İlerleme: 140/15852 şehir tarandı. (Başarılı: 140)
İlerleme: 160/15852 şehir tarandı. (Başarılı: 160)
İlerleme: 180/15852 şehir tarandı. (Başarılı: 178)
İlerleme: 200/15852 şehir tarandı. (Başarılı: 198)
İlerleme: 220/15852 şehir tarandı. (Başarılı: 218)
İlerleme: 240/15852 şehir tarandı. (Başarılı: 237)
İlerleme: 260/15852 şehir tarandı. (Başarılı: 257)
İlerleme: 280/15852 şehir tarandı. (Başarılı: 276)
İlerleme: 300/15852 şehir tarandı. (Başarılı: 296)
İlerleme: 320/15852 şehir tarandı. (Başarılı: 316)
İlerleme: 340/15852 şehir tarandı.

İlerleme: 1260/15852 şehir tarandı. (Başarılı: 1192)
İlerleme: 1280/15852 şehir tarandı. (Başarılı: 1212)
İlerleme: 1300/15852 şehir tarandı. (Başarılı: 1230)
İlerleme: 1320/15852 şehir tarandı. (Başarılı: 1249)
İlerleme: 1340/15852 şehir tarandı. (Başarılı: 1268)
İlerleme: 1360/15852 şehir tarandı. (Başarılı: 1286)
İlerleme: 1380/15852 şehir tarandı. (Başarılı: 1305)
İlerleme: 1400/15852 şehir tarandı. (Başarılı: 1323)
İlerleme: 1420/15852 şehir tarandı. (Başarılı: 1338)
İlerleme: 1440/15852 şehir tarandı. (Başarılı: 1356)
İlerleme: 1460/15852 şehir tarandı. (Başarılı: 1376)
İlerleme: 1480/15852 şehir tarandı. (Başarılı: 1394)
İlerleme: 1500/15852 şehir tarandı. (Başarılı: 1413)
İlerleme: 1520/15852 şehir tarandı. (Başarılı: 1432)
İlerleme: 1540/15852 şehir tarandı. (Başarılı: 1451)
İlerleme: 1560/15852 şehir tarandı. (Başarılı: 1471)
İlerleme: 1580/15852 şehir tarandı. (Başarılı: 1488)
İlerleme: 1600/15852 şehir tarandı. (Başarılı: 1507)
İlerleme: 1620/15852 şehir tarandı. (Başarılı:

İlerleme: 2220/15852 şehir tarandı. (Başarılı: 2085)
İlerleme: 2240/15852 şehir tarandı. (Başarılı: 2104)
İlerleme: 2260/15852 şehir tarandı. (Başarılı: 2121)
İlerleme: 2280/15852 şehir tarandı. (Başarılı: 2141)
İlerleme: 2300/15852 şehir tarandı. (Başarılı: 2160)
İlerleme: 2320/15852 şehir tarandı. (Başarılı: 2177)
İlerleme: 2340/15852 şehir tarandı. (Başarılı: 2197)
İlerleme: 2360/15852 şehir tarandı. (Başarılı: 2216)
İlerleme: 2380/15852 şehir tarandı. (Başarılı: 2234)
İlerleme: 2400/15852 şehir tarandı. (Başarılı: 2253)
İlerleme: 2420/15852 şehir tarandı. (Başarılı: 2269)
İlerleme: 2440/15852 şehir tarandı. (Başarılı: 2286)
İlerleme: 2460/15852 şehir tarandı. (Başarılı: 2303)
İlerleme: 2480/15852 şehir tarandı. (Başarılı: 2319)
İlerleme: 2500/15852 şehir tarandı. (Başarılı: 2336)
İlerleme: 2520/15852 şehir tarandı. (Başarılı: 2355)
İlerleme: 2540/15852 şehir tarandı. (Başarılı: 2374)
İlerleme: 2560/15852 şehir tarandı. (Başarılı: 2393)
İlerleme: 2580/15852 şehir tarandı. (Başarılı:

İlerleme: 2780/15852 şehir tarandı. (Başarılı: 2594)
İlerleme: 2800/15852 şehir tarandı. (Başarılı: 2612)
İlerleme: 2820/15852 şehir tarandı. (Başarılı: 2630)
İlerleme: 2840/15852 şehir tarandı. (Başarılı: 2649)
İlerleme: 2860/15852 şehir tarandı. (Başarılı: 2665)
İlerleme: 2880/15852 şehir tarandı. (Başarılı: 2685)
İlerleme: 2900/15852 şehir tarandı. (Başarılı: 2704)
İlerleme: 2920/15852 şehir tarandı. (Başarılı: 2724)
İlerleme: 2940/15852 şehir tarandı. (Başarılı: 2743)
İlerleme: 2960/15852 şehir tarandı. (Başarılı: 2762)
İlerleme: 2980/15852 şehir tarandı. (Başarılı: 2781)
İlerleme: 3000/15852 şehir tarandı. (Başarılı: 2801)
İlerleme: 3020/15852 şehir tarandı. (Başarılı: 2820)
İlerleme: 3040/15852 şehir tarandı. (Başarılı: 2838)
İlerleme: 3060/15852 şehir tarandı. (Başarılı: 2857)
İlerleme: 3080/15852 şehir tarandı. (Başarılı: 2876)
İlerleme: 3100/15852 şehir tarandı. (Başarılı: 2895)


İlerleme: 3120/15852 şehir tarandı. (Başarılı: 2912)
İlerleme: 3140/15852 şehir tarandı. (Başarılı: 2931)
İlerleme: 3160/15852 şehir tarandı. (Başarılı: 2951)
İlerleme: 3180/15852 şehir tarandı. (Başarılı: 2969)
İlerleme: 3200/15852 şehir tarandı. (Başarılı: 2988)
İlerleme: 3220/15852 şehir tarandı. (Başarılı: 3008)
İlerleme: 3240/15852 şehir tarandı. (Başarılı: 3025)
İlerleme: 3260/15852 şehir tarandı. (Başarılı: 3043)
İlerleme: 3280/15852 şehir tarandı. (Başarılı: 3062)
İlerleme: 3300/15852 şehir tarandı. (Başarılı: 3080)
İlerleme: 3320/15852 şehir tarandı. (Başarılı: 3100)
İlerleme: 3340/15852 şehir tarandı. (Başarılı: 3118)
İlerleme: 3360/15852 şehir tarandı. (Başarılı: 3138)
İlerleme: 3380/15852 şehir tarandı. (Başarılı: 3157)
İlerleme: 3400/15852 şehir tarandı. (Başarılı: 3175)
İlerleme: 3420/15852 şehir tarandı. (Başarılı: 3195)
İlerleme: 3440/15852 şehir tarandı. (Başarılı: 3215)
İlerleme: 3460/15852 şehir tarandı. (Başarılı: 3233)
İlerleme: 3480/15852 şehir tarandı. (Başarılı:

...

İlerleme: 15080/15852 şehir tarandı. (Başarılı: 14299)
İlerleme: 15100/15852 şehir tarandı. (Başarılı: 14317)
İlerleme: 15120/15852 şehir tarandı. (Başarılı: 14336)
İlerleme: 15140/15852 şehir tarandı. (Başarılı: 14356)
İlerleme: 15160/15852 şehir tarandı. (Başarılı: 14376)
İlerleme: 15180/15852 şehir tarandı. (Başarılı: 14394)
İlerleme: 15200/15852 şehir tarandı. (Başarılı: 14414)
İlerleme: 15220/15852 şehir tarandı. (Başarılı: 14433)
İlerleme: 15240/15852 şehir tarandı. (Başarılı: 14453)
İlerleme: 15260/15852 şehir tarandı. (Başarılı: 14473)
İlerleme: 15280/15852 şehir tarandı. (Başarılı: 14492)
İlerleme: 15300/15852 şehir tarandı. (Başarılı: 14512)
İlerleme: 15320/15852 şehir tarandı. (Başarılı: 14531)
İlerleme: 15340/15852 şehir tarandı. (Başarılı: 14549)
İlerleme: 15360/15852 şehir tarandı. (Başarılı: 14567)
İlerleme: 15380/15852 şehir tarandı. (Başarılı: 14585)
İlerleme: 15400/15852 şehir tarandı. (Başarılı: 14602)
İlerleme: 15420/15852 şehir tarandı. (Başarılı: 14622)
İlerleme: 

İlerleme: 15460/15852 şehir tarandı. (Başarılı: 14658)
İlerleme: 15480/15852 şehir tarandı. (Başarılı: 14678)
İlerleme: 15500/15852 şehir tarandı. (Başarılı: 14697)
İlerleme: 15520/15852 şehir tarandı. (Başarılı: 14714)
İlerleme: 15540/15852 şehir tarandı. (Başarılı: 14734)
İlerleme: 15560/15852 şehir tarandı. (Başarılı: 14752)
İlerleme: 15580/15852 şehir tarandı. (Başarılı: 14769)
İlerleme: 15600/15852 şehir tarandı. (Başarılı: 14787)
İlerleme: 15620/15852 şehir tarandı. (Başarılı: 14806)


İlerleme: 15640/15852 şehir tarandı. (Başarılı: 14823)
İlerleme: 15660/15852 şehir tarandı. (Başarılı: 14842)


İlerleme: 15680/15852 şehir tarandı. (Başarılı: 14859)
İlerleme: 15700/15852 şehir tarandı. (Başarılı: 14878)
İlerleme: 15720/15852 şehir tarandı. (Başarılı: 14896)
İlerleme: 15740/15852 şehir tarandı. (Başarılı: 14915)
İlerleme: 15760/15852 şehir tarandı. (Başarılı: 14935)
İlerleme: 15780/15852 şehir tarandı. (Başarılı: 14955)
İlerleme: 15800/15852 şehir tarandı. (Başarılı: 14974)


İlerleme: 15820/15852 şehir tarandı. (Başarılı: 14991)
İlerleme: 15840/15852 şehir tarandı. (Başarılı: 15010)
İlerleme: 15852/15852 şehir tarandı. (Başarılı: 15022)

HATALI ŞEHİRLER İŞLEMİ TAMAMLANDI
Toplam Taranan (Hatalı): 15852
Başarılı Veri (Şehir Bazlı): 15022
Hatalar 'hatali_sehirler_log.txt' dosyasına kaydedildi.

İlk 20 satır:
         time station_id     city_id city_name country  tavg  tmin  tmax  \
0  2012-01-01      58040  1156086320     Linyi      CN  -1.1  -5.2   4.1   
1  2012-02-01      58040  1156086320     Linyi      CN   0.2  -3.6   4.6   
2  2012-03-01      58040  1156086320     Linyi      CN   5.5   1.6  10.3   
3  2012-04-01      58040  1156086320     Linyi      CN  14.7   9.9  20.1   
4  2012-05-01      58040  1156086320     Linyi      CN  20.8  15.5  25.9   
5  2012-06-01      58040  1156086320     Linyi      CN  23.6  20.5  27.4   
6  2012-07-01      58040  1156086320     Linyi      CN  27.7  24.7  31.1   
7  2012-08-01      58040  1156086320     Linyi      CN 

In [28]:
import pandas as pd
import os

# Dosya isimleri
file_main = "tum_dunya_hava_verileri.csv"
file_error = "hatali_sehirler_hava_verileri.csv"

# Çıkacak birleşik dosya
output_combined = "tum_hava_verileri_birlesik.csv"

# Dosyaların varlığını kontrol et
if not os.path.exists(file_main):
    print(f"HATA: Ana veri dosyası bulunamadı: {file_main}")
    raise SystemExit

if not os.path.exists(file_error):
    print(f"HATA: Hatalı veri dosyası bulunamadı: {file_error}")
    raise SystemExit

print("CSV dosyaları okunuyor...")

# Dosyaları yükle
df_main = pd.read_csv(file_main)
df_error = pd.read_csv(file_error)

print("Birleştiriliyor (concat)...")

# Concat — alt alta birleştir
df_combined = pd.concat([df_main, df_error], ignore_index=True)

print("Toplam satır sayısı:", len(df_combined))

# Son halini kaydet
df_combined.to_csv(output_combined, index=False)

print(f"Başarılı! Birleşik dosya oluşturuldu: {output_combined}")

CSV dosyaları okunuyor...
Birleştiriliyor (concat)...
Toplam satır sayısı: 5940013
Başarılı! Birleşik dosya oluşturuldu: tum_hava_verileri_birlesik.csv


In [29]:
df_combined

,time,city_id,city_name,country,tavg,tmin,tmax,lat,lng,station_id
0,2012-01-01,1392685764,Tokyo,JP,4.8,1.8,8.3,35.6870,139.7495,NaN
1,2012-02-01,1392685764,Tokyo,JP,5.4,2.2,9.1,35.6870,139.7495,NaN
2,2012-03-01,1392685764,Tokyo,JP,8.8,5.3,12.5,35.6870,139.7495,NaN
3,2012-04-01,1392685764,Tokyo,JP,14.5,11.0,18.5,35.6870,139.7495,NaN
4,2012-05-01,1392685764,Tokyo,JP,19.6,16.1,23.6,35.6870,139.7495,NaN
...,...,...,...,...,...,...,...,...,...,...
5940008,2025-06-01,1716206606,Lupane,ZW,15.9,8.4,24.5,-18.9315,27.8070,67853
5940009,2025-07-01,1716206606,Lupane,ZW,16.3,9.1,24.5,-18.9315,27.8070,67853
5940010,2025-08-01,1716206606,Lupane,ZW,18.8,10.0,28.1,-18.9315,27.8070,67853
5940011,2025-09-01,1716206606,Lupane,ZW,24.5,16.2,32.9,-18.9315,27.8070,67853


In [5]:
import pandas as pd
import os
from datetime import datetime
from meteostat import Stations, Monthly

# ---------------------------------------------------------
# 1. AYARLAR
# ---------------------------------------------------------

file_path = 'worldcities.csv'
log_file_path = 'hatali_sehirler_log.txt'
output_csv_path = 'tum_dunya_hava_verileri.csv'  # Sonuçların kaydedileceği dosya

# Tarih Aralığı
start = datetime(1951, 1, 1)
end = datetime.now()

# ---------------------------------------------------------
# 2. LOGLAMA FONKSİYONU
# ---------------------------------------------------------
def log_error(city_row, error_message):
    timestamp = datetime.now().strftime("%Y-%m-%d %H:%M:%S")
    # CSV'deki kolon isimlerine göre loglama
    log_line = (
        f"{timestamp} | ID: {city_row['id']} | City: {city_row['city']} | "
        f"Country: {city_row['iso2']} | Error: {error_message}\n"
    )
    
    try:
        with open(log_file_path, "a", encoding="utf-8") as f:
            f.write(log_line)
    except Exception as e:
        print(f"Log dosyasına yazılamadı: {e}")

# ---------------------------------------------------------
# 3. VERİYİ OKUMA VE İŞLEME
# ---------------------------------------------------------

# Log dosyasını sıfırla
with open(log_file_path, "w", encoding="utf-8") as f:
    f.write(f"--- İŞLEM BAŞLANGICI: {datetime.now()} ---\n")
    f.write("Log Formatı: ZAMAN | ID | ŞEHİR | ÜLKE KODU | HATA\n")
    f.write("-" * 80 + "\n")

all_weather_data = []

if not os.path.exists(file_path):
    print(f"HATA: Dosya bulunamadı: {file_path}")
else:
    print(f"CSV okunuyor: {file_path}...")
    
    # CSV dosyasını Pandas ile oku
    df_cities = pd.read_csv(file_path)
    
    # Tüm veri seti üzerinde çalışacağız
    total_cities = len(df_cities)
    print(f"Toplam Şehir Sayısı (Tüm Dünya): {total_cities}")
    print("Veri çekme işlemi başlıyor... (Bu işlem uzun sürebilir)\n")

    count = 0
    success_count = 0
    
    # Tüm DataFrame üzerinde döngü
    for index, row in df_cities.iterrows():
        try:
            # CSV kolon isimleri: city, lat, lng, iso2, id
            lat = row['lat']
            lng = row['lng']
            city_name = row['city']
            country_iso = row['iso2']
            city_id = row['id']

            # --- BURASI DEĞİŞTİ: Point yerine Stations().nearby() kullanılıyor ---

            # En yakın istasyonu bul
            stations = Stations()
            stations = stations.nearby(lat, lng)

            # Tek bir istasyon seç (en yakın)
            station = stations.fetch(1)

            if station.empty:
                # Hiç istasyon bulunamadıysa logla ve devam et
                log_error(row, "Yakın istasyon bulunamadı")
                count += 1
                if count % 100 == 0:
                    print(f"İlerleme: {count}/{total_cities} şehir tarandı. (Başarılı: {success_count})")
                continue

            # station DataFrame'ini location olarak kullan
            data = Monthly(station, start, end)
            data = data.fetch()

            if not data.empty:
                # İstersen istasyon ID'sini de ekleyebilirsin (index üzerinden)
                station_id = station.index[0]
                data['station_id'] = station_id

                # Meta verileri ekle
                data['city_id'] = city_id
                data['city_name'] = city_name
                data['country'] = country_iso
                data['lat'] = lat
                data['lng'] = lng
                
                # İndeksi sütuna çevir (time sütunu oluşur)
                data.reset_index(inplace=True)
                
                # İstenen sütunları seç (pointer kodundaki ile aynı yapıldı)
                # Eğer station_id'yi de CSV'de görmek istersen, aşağıdaki listeye 'station_id' ekleyebilirsin.
                selected_cols = data[
                    ['time', 'city_id', 'city_name', 'country', 'tavg', 'tmin', 'tmax', 'lat', 'lng']
                ]
                all_weather_data.append(selected_cols)
                
                success_count += 1
            else:
                # Meteostat veri döndüremedi
                log_error(row, "Meteostat verisi boş (Empty Data - station)")

        except Exception as e:
            # Beklenmeyen bir hata
            log_error(row, f"Hata: {str(e)}")
        
        count += 1
        
        # İlerleme durumunu göster (Her 100 şehirde bir yazdıralım, liste uzun olduğu için)
        if count % 100 == 0:
            print(f"İlerleme: {count}/{total_cities} şehir tarandı. (Başarılı: {success_count})")

    # ---------------------------------------------------------
    # 4. SONUCU GÖSTERME VE KAYDETME
    # ---------------------------------------------------------

    print("\n" + "="*50)
    print("İŞLEM TAMAMLANDI")
    print(f"Toplam Taranan: {count}")
    print(f"Başarılı Veri (Şehir Bazlı): {success_count}")
    print(f"Hatalar '{log_file_path}' dosyasına kaydedildi.")
    print("="*50)

    if all_weather_data:
        # Tüm verileri birleştir
        final_df = pd.concat(all_weather_data, ignore_index=True)
        
        print("\nSONUÇ DATAFRAME (İLK 20 SATIR):")
        print(final_df.head(20))
        
        print("\nDATAFRAME BİLGİSİ:")
        print(final_df.info())

        # Veriyi CSV olarak kaydet
        print(f"\nVeriler '{output_csv_path}' dosyasına kaydediliyor...")
        final_df.to_csv(output_csv_path, index=False)
        print("Kayıt işlemi başarılı.")
        
    else:
        print("Hiçbir veri çekilemedi.")

CSV okunuyor: worldcities.csv...
Toplam Şehir Sayısı (Tüm Dünya): 48059
Veri çekme işlemi başlıyor... (Bu işlem uzun sürebilir)

İlerleme: 100/48059 şehir tarandı. (Başarılı: 99)
İlerleme: 200/48059 şehir tarandı. (Başarılı: 198)
İlerleme: 300/48059 şehir tarandı. (Başarılı: 297)
İlerleme: 400/48059 şehir tarandı. (Başarılı: 397)


İlerleme: 500/48059 şehir tarandı. (Başarılı: 496)
İlerleme: 600/48059 şehir tarandı. (Başarılı: 596)


İlerleme: 700/48059 şehir tarandı. (Başarılı: 692)


İlerleme: 800/48059 şehir tarandı. (Başarılı: 789)


İlerleme: 900/48059 şehir tarandı. (Başarılı: 888)


...

İlerleme: 47600/48059 şehir tarandı. (Başarılı: 46438)


İlerleme: 47700/48059 şehir tarandı. (Başarılı: 46535)


İlerleme: 47800/48059 şehir tarandı. (Başarılı: 46629)


İlerleme: 47900/48059 şehir tarandı. (Başarılı: 46727)


İlerleme: 48000/48059 şehir tarandı. (Başarılı: 46823)

İŞLEM TAMAMLANDI
Toplam Taranan: 48059
Başarılı Veri (Şehir Bazlı): 46881
Hatalar 'hatali_sehirler_log.txt' dosyasına kaydedildi.

SONUÇ DATAFRAME (İLK 20 SATIR):
         time     city_id city_name country  tavg  tmin  tmax     lat  \
0  1951-01-01  1392685764     Tokyo      JP   3.3  -1.0   8.6  35.687   
1  1951-02-01  1392685764     Tokyo      JP   4.5   0.4   9.7  35.687   
2  1951-03-01  1392685764     Tokyo      JP   8.8   4.1  14.2  35.687   
3  1951-04-01  1392685764     Tokyo      JP  13.3   9.0  18.3  35.687   
4  1951-05-01  1392685764     Tokyo      JP  18.0  13.9  23.3  35.687   
5  1951-06-01  1392685764     Tokyo      JP  21.2  17.3  25.9  35.687   
6  1951-07-01  1392685764     Tokyo      JP  24.3  21.2  28.3  35.687   
7  1951-08-01  1392685764     Tokyo      JP  26.7  23.5  31.6  35.687   
8  1951-09-01  1392685764     Tokyo      JP  20.7  17.4  24.7  35.687   
9  1951-10-01  1392685764     Tokyo      JP  17.3  

In [6]:
final_df

,time,city_id,city_name,country,tavg,tmin,tmax,lat,lng
0,1951-01-01,1392685764,Tokyo,JP,3.3,-1.0,8.6,35.687,139.7495
1,1951-02-01,1392685764,Tokyo,JP,4.5,0.4,9.7,35.687,139.7495
2,1951-03-01,1392685764,Tokyo,JP,8.8,4.1,14.2,35.687,139.7495
3,1951-04-01,1392685764,Tokyo,JP,13.3,9.0,18.3,35.687,139.7495
4,1951-05-01,1392685764,Tokyo,JP,18.0,13.9,23.3,35.687,139.7495
...,...,...,...,...,...,...,...,...,...
17389092,2025-06-01,1850037473,Charlotte Amalie,VI,29.0,26.6,32.4,18.342,-64.9331
17389093,2025-07-01,1850037473,Charlotte Amalie,VI,29.1,26.5,32.6,18.342,-64.9331
17389094,2025-08-01,1850037473,Charlotte Amalie,VI,29.9,27.5,33.0,18.342,-64.9331
17389095,2025-09-01,1850037473,Charlotte Amalie,VI,29.5,27.3,32.3,18.342,-64.9331
